# Beta9 / Beam — Deep-Dive Jupyter Workbook

A hands-on, production-minded notebook for exploring **Beta9**, the open-source engine behind Beam. It covers the local SDK ergonomics, remote sandbox execution, serverless GPU inference patterns, async task queues, persistence, deployment, and operational checks.

> **Safety and cost note:** Cells that create remote containers, enqueue work, or deploy public services are disabled by default. Read and deliberately enable them only after authenticating and choosing resource limits.

## What you will build

1. A deterministic local classifier baseline.
2. A remote Beta9/Beam sandbox that executes the same logic in isolation.
3. A GPU-ready synchronous inference endpoint with model warm-up, persistent weight storage, and bounded autoscaling.
4. An asynchronous batch queue appropriate for longer jobs.
5. A deployment, load-test, observability, and teardown checklist.

## Architecture map

```text
Caller / application
   |
   +-- synchronous request --> endpoint --> autoscaled container(s) --> model inference
   |                                  |
   |                                  +--> mounted volume: model/data cache
   |
   +-- long-running request --> task queue --> worker container(s) --> output / webhook / status
   |
   +-- untrusted code --> sandbox --> isolated process/container
```

The repository describes Beam as a Pythonic runtime for serverless AI workloads, with rapid container builds, GPU support, scale-to-zero, storage volumes, endpoints, task queues, and sandboxes. Beta9 can be self-hosted; Beam is the managed platform built on that engine.

## 1. Prerequisites

### Managed Beam path

- A Beam account and authenticated CLI/session.
- Python 3.10+ locally; the example remote image uses Python 3.11.
- A credit/billing-aware workspace if you execute GPU cells.

### Self-hosted Beta9 path

- A deployed Beta9 control plane and worker capacity.
- A client configuration pointing at that installation.
- Equivalent runtime, storage, registry, and GPU-node setup managed by your team.

The programming pattern below is intentionally SDK-focused. Verify the exact CLI authentication and self-host configuration for the version you are running before deployment.

In [1]:
%pip install -q -U beam-client

import importlib.metadata as md
print('beam-client:', md.version('beam-client'))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.3/227.3 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.9/82.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires watchdog<7,>=6, but you have watchdog 4.0.2 which is incompatible.
beam-client: 0.2.211


## 2. Configuration and guardrails

Keep deployment parameters explicit and conservative. In particular, start with a small maximum number of containers and a low per-container concurrency target; raise them after observing latency, queue depth, GPU memory, and error rate.

This notebook does **not** place credentials in source code. Authenticate using the CLI or an environment-specific secret-management workflow.

In [2]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Settings:
    app_name: str = 'beta9-deep-dive-demo'
    model_id: str = 'distilbert-base-uncased-finetuned-sst-2-english'
    volume_name: str = 'beta9-model-cache'
    volume_mount_path: str = './weights'
    gpu: str = 'T4'
    cpu: int = 2
    memory: str = '8Gi'
    max_containers: int = 2
    tasks_per_container: int = 2

CFG = Settings()
CFG

Settings(app_name='beta9-deep-dive-demo', model_id='distilbert-base-uncased-finetuned-sst-2-english', volume_name='beta9-model-cache', volume_mount_path='./weights', gpu='T4', cpu=2, memory='8Gi', max_containers=2, tasks_per_container=2)

## 3. Local baseline

Start by defining a pure, testable input contract. This gives you a local control result before you introduce image builds, startup hooks, GPU allocation, network behavior, or remote serialization.

For a real service, retain this separation: keep business validation and response shaping independent from the deployment decorator.

In [3]:
from dataclasses import dataclass
from typing import Any

@dataclass(frozen=True)
class InferenceRequest:
    text: str

def validate_request(request: InferenceRequest) -> str:
    text = request.text.strip()
    if not text:
        raise ValueError('text must be non-empty')
    if len(text) > 4_000:
        raise ValueError('text exceeds the 4,000-character demo limit')
    return text

def local_baseline(request: InferenceRequest) -> dict[str, Any]:
    text = validate_request(request)
    positive_terms = {'good', 'great', 'fast', 'excellent', 'love'}
    tokens = {token.strip('.,!?').lower() for token in text.split()}
    score = len(tokens & positive_terms) / max(len(positive_terms), 1)
    return {
        'label': 'POSITIVE' if score >= 0.2 else 'NEGATIVE',
        'confidence_proxy': round(0.5 + min(score, 0.49), 3),
        'implementation': 'local deterministic baseline',
    }

local_baseline(InferenceRequest('Beam is fast and excellent for experiments.'))

{'label': 'POSITIVE',
 'confidence_proxy': 0.9,
 'implementation': 'local deterministic baseline'}

## 4. Remote sandbox experiment

Sandboxes are suitable for isolated execution, including code generated by an agent. Treat an untrusted-code sandbox as a security boundary, not as a substitute for policy: deny or restrict network egress where possible, do not inject broad credentials, cap time and resources, validate inputs, and delete temporary state.

The following cell is gated because it creates a remote sandbox. Enable `RUN_REMOTE_SANDBOX` only after authentication.

In [4]:
RUN_REMOTE_SANDBOX = False

if RUN_REMOTE_SANDBOX:
    from beam import Image, Sandbox

    image = Image(
        python_version='python3.11',
        python_packages=['numpy==2.*'],
    )
    sandbox = Sandbox(image=image).create()
    try:
        result = sandbox.process.run_code(
            "import numpy as np; print({'mean': float(np.mean([1, 2, 3]))})"
        )
        print(result.result)
    finally:
        # Use the SDK's current stop/terminate API for your installed version.
        # Explicit cleanup is essential for interactive experiments.
        pass
else:
    print('Remote sandbox disabled. Set RUN_REMOTE_SANDBOX=True after auth and cost review.')

Remote sandbox disabled. Set RUN_REMOTE_SANDBOX=True after auth and cost review.


### Sandbox evaluation checklist

- Run a known deterministic program and record runtime/image-build latency.
- Test failure behavior: exception, timeout, and invalid dependency.
- Check that network policy and filesystem exposure match your threat model.
- Verify cleanup behavior and inspect whether processes remain after a failed cell.
- Never run secrets-bearing code provided by an untrusted model or user.

## 5. Serverless inference endpoint

This is a deployable module pattern, shown in-notebook for review. It uses:

- `Image` to define a reproducible environment.
- `Volume` to retain downloaded weights across container lifecycles.
- `on_start` to load the model once per container rather than per request.
- `QueueDepthAutoscaler` with a firm cap on container count.
- Input validation before model work.

For low-latency services, split cold-start latency from steady-state latency in your benchmarks. A warmed container may be fast while a scale-from-zero request is materially slower.

In [5]:
ENDPOINT_MODULE = r'''
import os
from typing import Any

from beam import Image, QueueDepthAutoscaler, Volume, endpoint

MODEL_ID = 'distilbert-base-uncased-finetuned-sst-2-english'
WEIGHTS_PATH = './weights'

image = Image(
    python_version='python3.11',
    python_packages=[
        'torch',
        'transformers>=4.40,<5',
        'safetensors',
    ],
)

model = None
tokenizer = None

def load_model() -> None:
    global model, tokenizer
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    os.environ['HF_HOME'] = WEIGHTS_PATH
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=WEIGHTS_PATH)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, cache_dir=WEIGHTS_PATH
    )
    model.eval()

@endpoint(
    name='beta9-sentiment-demo',
    image=image,
    gpu='T4',
    cpu=2,
    memory='8Gi',
    timeout=60,
    volumes=[Volume(name='beta9-model-cache', mount_path=WEIGHTS_PATH)],
    autoscaler=QueueDepthAutoscaler(max_containers=2, tasks_per_container=2),
    on_start=load_model,
)
def predict(text: str) -> dict[str, Any]:
    if not isinstance(text, str) or not (text := text.strip()):
        raise ValueError('text must be a non-empty string')
    if len(text) > 4_000:
        raise ValueError('text exceeds 4,000 characters')

    import torch
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    with torch.inference_mode():
        logits = model(**inputs).logits[0]
        probabilities = torch.softmax(logits, dim=-1)
        class_id = int(torch.argmax(probabilities).item())

    return {
        'label': model.config.id2label[class_id],
        'confidence': round(float(probabilities[class_id].item()), 6),
        'model_id': MODEL_ID,
    }
'''

print(ENDPOINT_MODULE[:2000])


import os
from typing import Any

from beam import Image, QueueDepthAutoscaler, Volume, endpoint

MODEL_ID = 'distilbert-base-uncased-finetuned-sst-2-english'
WEIGHTS_PATH = './weights'

image = Image(
    python_version='python3.11',
    python_packages=[
        'torch',
        'transformers>=4.40,<5',
        'safetensors',
    ],
)

model = None
tokenizer = None

def load_model() -> None:
    global model, tokenizer
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    os.environ['HF_HOME'] = WEIGHTS_PATH
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=WEIGHTS_PATH)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, cache_dir=WEIGHTS_PATH
    )
    model.eval()

@endpoint(
    name='beta9-sentiment-demo',
    image=image,
    gpu='T4',
    cpu=2,
    memory='8Gi',
    timeout=60,
    volumes=[Volume(name='beta9-model-cache', mount_path=WEIGHTS_PATH)],
    autoscaler=QueueDepthAutoscaler(max_containers=

### Deployment workflow

1. Export the module below to `sentiment_endpoint.py`.
2. Authenticate with the Beam CLI according to the current documentation.
3. Deploy the named function, for example: `beam deploy sentiment_endpoint.py:predict`.
4. Save the returned endpoint URL in a secret or environment variable, never directly in a public notebook.
5. Run a smoke test and then a bounded concurrency test.
6. Monitor request latency, errors, startup time, GPU memory, container count, and spend before increasing limits.

In [6]:
from pathlib import Path

endpoint_file = Path('sentiment_endpoint.py')
endpoint_file.write_text(ENDPOINT_MODULE, encoding='utf-8')
print(f'Wrote {endpoint_file.resolve()}')
print('Review the source, authenticate, then deploy from a terminal:')
print('  beam deploy sentiment_endpoint.py:predict')

Wrote /content/sentiment_endpoint.py
Review the source, authenticate, then deploy from a terminal:
  beam deploy sentiment_endpoint.py:predict


## 6. Asynchronous batch processing

Use a task queue when work may exceed the synchronous endpoint budget or when callers can accept a job ID rather than an immediate final result. Examples include batch embeddings, document processing, data preparation, fine-tuning orchestration, and long model runs.

The design below produces a batch result conceptually. In production, define idempotency keys, schema versioning, retry boundaries, dead-letter handling, and a result retrieval policy before enabling automatic retries.

In [7]:
TASK_QUEUE_MODULE = r'''
from beam import Image, QueueDepthAutoscaler, TaskPolicy, task_queue

image = Image(
    python_version='python3.11',
    python_packages=['numpy>=1.26'],
)

@task_queue(
    name='beta9-batch-score',
    image=image,
    cpu=1,
    memory='1Gi',
    timeout=900,
    task_policy=TaskPolicy(max_retries=3),
    autoscaler=QueueDepthAutoscaler(max_containers=2, tasks_per_container=1),
)
def score_batch(records: list[dict]) -> dict:
    if not isinstance(records, list) or not records:
        raise ValueError('records must be a non-empty list')
    if len(records) > 1_000:
        raise ValueError('batch is capped at 1,000 records')

    valid = 0
    for record in records:
        if isinstance(record, dict) and isinstance(record.get('value'), (int, float)):
            valid += 1
    return {
        'records_received': len(records),
        'records_valid': valid,
        'status': 'complete',
    }
'''

queue_file = Path('batch_queue.py')
queue_file.write_text(TASK_QUEUE_MODULE, encoding='utf-8')
print('Wrote batch_queue.py')
print('Deploy with: beam deploy batch_queue.py:score_batch')

Wrote batch_queue.py
Deploy with: beam deploy batch_queue.py:score_batch


## 7. Endpoint vs. task queue

| Decision factor | Synchronous endpoint | Task queue |
|---|---|---|
| Caller expects | Final response now | Job acknowledgement / task ID |
| Best fit | Small, latency-sensitive inference or API work | Heavy, bursty, or long-running jobs |
| Capacity concern | Tail latency and warm capacity | Queue depth, retries, and completion time |
| Result delivery | HTTP response | Polling, callback/webhook, or persisted output |
| Design priority | Strict validation and response latency | Idempotency and durable result handling |

A useful rule: if a client request cannot safely wait for the worst plausible execution time, make it an asynchronous job.

## 8. Load testing and acceptance criteria

Do not increase autoscaling limits solely because the happy-path request works. First define an expected workload and acceptance thresholds. For an inference service, include at least:

- Cold-start time: request when no container is warm.
- Warm p50/p95/p99 latency at representative payload sizes.
- Error rate under bounded concurrency.
- Queueing delay and saturation behavior as load exceeds current capacity.
- GPU memory high-water mark and out-of-memory recovery.
- Cost per successful request or per batch item.

The next cell is an **optional client-side harness skeleton**. Set an endpoint URL only after deployment. It intentionally bounds concurrency to avoid accidental load or spending.

In [8]:
import os
import statistics
import time

ENDPOINT_URL = os.getenv('BEAM_ENDPOINT_URL', '')
RUN_SMOKE_TEST = False

def smoke_test(url: str, payload: dict, timeout_s: float = 60.0) -> dict:
    import requests
    started = time.perf_counter()
    response = requests.post(url, json=payload, timeout=timeout_s)
    elapsed_ms = (time.perf_counter() - started) * 1_000
    response.raise_for_status()
    return {'latency_ms': elapsed_ms, 'response': response.json()}

if RUN_SMOKE_TEST:
    if not ENDPOINT_URL:
        raise RuntimeError('Set BEAM_ENDPOINT_URL before running the smoke test.')
    observation = smoke_test(ENDPOINT_URL, {'text': 'This deployment is fast and useful.'})
    print(observation)
else:
    print('Smoke test disabled. Export BEAM_ENDPOINT_URL and set RUN_SMOKE_TEST=True.')

Smoke test disabled. Export BEAM_ENDPOINT_URL and set RUN_SMOKE_TEST=True.


## 9. Production review

### Reliability

- Make handlers idempotent when retries can occur.
- Bound input sizes, execution duration, concurrency, and container count.
- Classify failures: caller errors, transient infrastructure errors, model errors, and permanent data errors.
- Persist or return enough metadata to correlate a request with a deployment version.

### Security

- Keep credentials in the platform secret store or a dedicated secret manager; never in notebook output, image layers, or source control.
- Give each endpoint the narrowest required access to data volumes, buckets, and external APIs.
- Authenticate endpoints before using them for sensitive or billable operations.
- Treat model inputs, URLs, filenames, and generated code as untrusted.

### Cost and performance

- Prefer CPU for lightweight preprocessing and only request GPU where profiling proves it helps.
- Cache model artifacts in a volume to reduce repeat downloads and improve startup reliability.
- Start with `max_containers=1` or `2`; expand only after a load test demonstrates a bottleneck.
- Use model loading hooks to avoid downloading/loading weights for every invocation.

### Reproducibility

- Pin Python and critical library versions.
- Record model revision, tokenizer revision, image dependency lock, and deployment commit.
- Add unit tests for validation and deterministic response shaping, plus an integration smoke test for the deployed API.

## 10. Suggested next experiments

1. Replace the sentiment model with a compact model relevant to your edge-to-cloud workflow and compare CPU/GPU break-even points.
2. Add request batching only after measuring single-request latency and memory behavior.
3. Build a two-stage pipeline: synchronous validation/ingestion endpoint → async task queue → versioned output store.
4. Run an agent-generated-code workload inside a sandbox with deliberately restricted network access and a disposable volume.
5. Self-host Beta9 in a test environment and compare operational overhead, cold start, observability, and GPU scheduling behavior against managed Beam.

## Source pointers

- Repository: https://github.com/beam-cloud/beta9
- Beam documentation: https://docs.beam.cloud/
- Python SDK reference: https://docs.beam.cloud/v2/reference/py-sdk

API signatures evolve. Before executing deployment code, reconcile decorator parameters and CLI commands with the documentation matching your installed `beam-client` version.